In [ ]:
import papermill as pm
import pandas as pd
from multiprocessing import Pool
import concurrent.futures
import queue
import os
from pathlib import Path
from datetime import datetime

In [ ]:
def run_notebook(parameters,notebook_to_run,parameters_common={}):
    
    # change to directory where the notebook is (resolve relative imports)
    os.chdir(Path(notebook_to_run).absolute().parent)
    
    # run notebook
    for parameters_spec in parameters_list:
        parameters = {**parameters_common, **parameters_spec}

        pm.execute_notebook(
           notebook_to_run,
            "/home/stumberger/image-analyis-recipes/alignment/correct_chromatic_aberration_for_tables_gs_copy.ipynb",
#            '/dev/null',
           parameters=parameters)

# 0) Create projections

In [ ]:
parameters_list = [

    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS614_Sox2_118kb_mESC/raw/"},
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/visualization/nd2_make_projections.ipynb"

run_notebook(parameters_list,notebook_to_run)

# 1) Resave .nd2 to tif 

In [ ]:
parameters_list = [
    
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS626_Prdm14_409kb_mESC/"}
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/resave/resave_nd2_as_tiff.ipynb"

run_notebook(parameters_list,notebook_to_run)

# 2) Spot detection

In [ ]:
parameters_list = [
    
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS162_Dppa3-58kb_EpiSC/"}

    ]

parameters_common = {"channels": [1,2],
                    "tif_subfolder": "tif",
                    "out_subfolder": "detections"}

notebook_to_run = "/home/stumberger/image-analyis-recipes/spot-detection/RS-FISH_spot_detection.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 3) Correct chromatic shift

In [ ]:
parameters_list = [

    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS162_Dppa3-58kb_EpiSC/"}


]

parameters_common = {"pixel_size": [0.3,0.13,0.13],
                     "channel_aliases" : {
                            '405-CSU-W1': '405-CSU-W1',
                            '488-CSU-W1': '488-CSU-W1',
                            '561-CSU-W1': 1,
                            '640-CSU-W1': 2},
                     "reference_channel": 1,
                     "csv_string" : "merge_subpixel.csv",
                     "transforms_path": "/data/agl_data/NanoFISH/Gabi/GS666_tetraspeck_on_cells_1-50/channel_registration_multifile1.json"
}

notebook_to_run = "/home/stumberger/image-analyis-recipes/alignment/correct_chromatic_aberration_for_tables_v1.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 4) Segment cells

In [ ]:
import papermill as pm
import concurrent.futures
import queue

def execute_notebook(parameters):
    pm.execute_notebook(
        '/home/stumberger/image-analyis-recipes/segmentation/cellpose_segmentation_3d.ipynb',
        None,
        parameters=parameters)

def process_parameters(parameters_queue, parameters_common):
    while not parameters_queue.empty():
        parameters_spec = parameters_queue.get()
        execute_notebook({**parameters_common, **parameters_spec})
        parameters_queue.task_done()

if __name__ == '__main__':
    parameters_common = {
        "model": "es_20231026"}

    parameters_list =     parameters_list = parameters_list = [
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS629_Prdm14_47kb_EpiSC/"}

]

    batch_size = 1  # Number of parallel tasks

    parameters_queue = queue.Queue()
    for parameters_spec in parameters_list:
        parameters_queue.put(parameters_spec)

    with concurrent.futures.ThreadPoolExecutor(max_workers=batch_size) as executor:
        # Start the initial batch
        initial_batch = [executor.submit(execute_notebook, {**parameters_common, **parameters_queue.get()}) for _ in range(batch_size)]
        
        # Continue processing new tasks as previous ones complete
        while not parameters_queue.empty():
            # Wait for any task in the initial batch to complete
            concurrent.futures.wait(initial_batch, return_when=concurrent.futures.FIRST_COMPLETED)
            
            # Replace completed tasks with new tasks
            completed = [task for task in initial_batch if task.done()]
            for task in completed:
                initial_batch.remove(task)
                new_task = executor.submit(execute_notebook, {**parameters_common, **parameters_queue.get()})
                initial_batch.append(new_task)

        # Wait for all tasks to complete
        concurrent.futures.wait(initial_batch)

    parameters_queue.join()

# 4.1) Add segmentation info to spots

In [ ]:
parameters_common = {
    "filter": True, # remove spots outsde of cells?
    "mask_ending": "_seg",
    "in_subpath": "detections",
    "spot_file": "merge_subpixel_shift-corrected.csv"}

parameters_list = [
    
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS162_Dppa3-58kb_EpiSC/"}

    
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/measurement/assign_spots_to_cell.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 4.2) Quality measure: calculate sensitivity

In [ ]:
parameters_common = {"mask_ending": "_seg",
                    "rel_spot_path": "/detections/merge_filtered.csv"} #spot path relative to upper folder}

parameters_list = [
    
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS648_Nanog_intron/20250409_sd/"}

]

notebook_to_run = "/home/stumberger/image-analyis-recipes/measurement/calculate_spots_per_cell.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 5) Calculate distances

In [ ]:
parameters_common = {
    "rel_spot_path": "/detections/merge_filtered.csv", #spot path relative to upper folder
        "channels": [1,2],
        "voxel_size": [300, 130, 130]} #spot path relative to upper folder

parameters_list = [

    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS162_Dppa3-58kb_EpiSC/"}

    
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/measurement/paired_spot_distances_2_channels.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 6) Join csvs

In [ ]:
wd = "/data/agl_data/NanoFISH/Gabi/"

csv_files = [

    "/data/agl_data/NanoFISH/Gabi/GS161_Dppa3-41kb_EpiSC/",
    "/data/agl_data/NanoFISH/Gabi/GS162_Dppa3-58kb_EpiSC/"
]

# Initialize an empty list to store DataFrames
dataframes = []

# Read and store each CSV file as a DataFrame
for file in csv_files:
    df = pd.read_csv(f"{file}/distances.csv")
    # df = pd.read_csv(f"{file}/detections/spots_per_cell.csv")
    dataframes.append(df)

# Join the DataFrames using Pandas (e.g., concatenate them vertically)
joined_dataframe = pd.concat(dataframes, ignore_index=True)

# Save the joined DataFrame to a new CSV file
time =  datetime.now().strftime("%Y-%m-%d-%H-%M")
joined_dataframe.to_csv(f'{wd}/all_distances_filtered_{time}.csv', index=False)
# joined_dataframe.to_csv(f'{wd}/all_spots_per_cell_{time}.csv', index=False)